# Log Clustering for Pattern Discovery — Loki Logs

Ce notebook applique le pipeline **TF-IDF + K-Means** (inspiré du Module 3 AIOps) sur les logs collectés depuis Loki.

**Pipeline :**
1. Chargement & exploration du dataset CSV
2. Nettoyage & extraction de patterns
3. Vectorisation TF-IDF
4. Exploration du vocabulaire
5. K-Means clustering
6. Choix du nombre optimal de clusters (Elbow + Silhouette)
7. Exploration du contenu des clusters
8. Labeling & interprétation
9. Documentation & export

## 1 — Chargement & Exploration des Logs

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

# Chemin absolu vers le projet
PROJECT_DIR = r'd:\monitoring-ia\ML\ML_Log_Loki'
CSV_PATH = os.path.join(PROJECT_DIR, 'datasets', 'mock_loki_logs.csv')
df_logs = pd.read_csv(CSV_PATH)

print('Dataset charge avec succes!')
print(f'  Nombre de lignes : {len(df_logs):,}')
print(f'  Colonnes : {list(df_logs.columns)}')
print()
df_logs.head(10)

In [ ]:
# Distribution des niveaux de log
print('Distribution des niveaux de log :')
print('-' * 40)
level_counts = df_logs['level'].value_counts()
for lvl, count in level_counts.items():
    pct = count / len(df_logs) * 100
    print(f'  {lvl:>8}: {count:>5} ({pct:.1f}%)')

print(f'\nDistribution des endpoints :')
print(df_logs['endpoint'].value_counts())

print(f'\nDistribution des status_code :')
print(df_logs['status_code'].value_counts())

## 2 — Nettoyage & Extraction de Patterns

On normalise les messages et on extrait un pattern opérationnel pour chaque log, comme dans le notebook de référence.

In [ ]:
import re

# Nettoyage : message en minuscule
df_logs['message_clean'] = df_logs['message'].astype(str).str.strip().str.lower()

def extract_log_pattern(row):
    """Extraire un pattern opérationnel structuré pour le clustering."""
    parts = []
    # Méthode HTTP
    parts.append(str(row['method']).lower())
    # Endpoint normalisé
    endpoint = str(row['endpoint']).lower()
    endpoint = re.sub(r'/\d+', '/{id}', endpoint)
    parts.append(endpoint.replace('/', '_').strip('_') or 'root')
    # Status code catégorisé
    sc = int(row['status_code'])
    parts.append(f'status_{sc}')
    # Composant
    parts.append(str(row['component']).lower())
    # Action
    parts.append(str(row['action']).lower())
    # Level
    parts.append(str(row['level']).lower())
    return ' '.join(parts)

df_logs['log_pattern'] = df_logs.apply(extract_log_pattern, axis=1)

print('Extraction de patterns terminée!')
print(f'  Patterns uniques : {df_logs["log_pattern"].nunique()}')
print(f'  Messages uniques : {df_logs["message_clean"].nunique()}')
reduction = (1 - df_logs['log_pattern'].nunique() / df_logs['message_clean'].nunique()) * 100
print(f'  Réduction : {reduction:.1f}%')
print()
print('Exemples de patterns :')
for i in range(min(5, len(df_logs))):
    print(f'  [{df_logs.iloc[i]["level"]}] {df_logs.iloc[i]["log_pattern"]}')

## 3 — Vectorisation TF-IDF

Conversion des patterns textuels en vecteurs numériques avec TF-IDF.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=100,
    stop_words='english',
    min_df=2,
    max_df=0.95,
    token_pattern=r'(?u)\b\w+\b'
)

X_tfidf = vectorizer.fit_transform(df_logs['log_pattern'])

print('TF-IDF terminé!')
print(f'  Matrice : {X_tfidf.shape}')
print(f'  Features : {X_tfidf.shape[1]} termes uniques')
print(f'  Documents : {X_tfidf.shape[0]:,} entrées')

feature_names = vectorizer.get_feature_names_out()
print(f'  Vocabulaire : {list(feature_names[:20])}')

## 4 — Exploration du Vocabulaire

In [ ]:
# Importance des termes
word_importance = np.asarray(X_tfidf.sum(axis=0)).flatten()
sorted_indices = np.argsort(word_importance)[::-1]
top_words_list = feature_names[sorted_indices[:15]]
top_values = word_importance[sorted_indices[:15]]

plt.figure(figsize=(10, 5))
plt.barh(top_words_list[::-1], top_values[::-1], color='steelblue', alpha=0.7)
plt.title('Top 15 Termes les Plus Importants (TF-IDF)', fontsize=14)
plt.xlabel('Poids TF-IDF Total')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 5 — Clustering K-Means

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df_logs['cluster_id'] = kmeans.fit_predict(X_tfidf)

cluster_counts = df_logs['cluster_id'].value_counts().sort_index()

print('Clustering terminé!')
print(f'\nDistribution des clusters :')
for cid, count in cluster_counts.items():
    pct = count / len(df_logs) * 100
    print(f'  Cluster {cid}: {count:>4} entrées ({pct:5.1f}%)')

df_logs[['message', 'level', 'endpoint', 'cluster_id']].head(10)

## 6 — Choix Optimal du Nombre de Clusters

Évaluation avec la méthode du coude (Elbow) et le score Silhouette.

In [ ]:
from sklearn.metrics import silhouette_score

inertias, silhouettes = [], []
k_values = [3, 5, 8, 10, 12, 15]

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    preds = model.fit_predict(X_tfidf)
    inertias.append(model.inertia_)
    sil = silhouette_score(X_tfidf, preds)
    silhouettes.append(sil)
    print(f'  k={k:2d}: inertia={model.inertia_:10.0f}, silhouette={sil:.3f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(k_values, inertias, 'o-', color='steelblue', linewidth=2, markersize=8)
ax1.set_title('Elbow Method', fontsize=13)
ax1.set_xlabel('Nombre de Clusters (k)')
ax1.set_ylabel('Inertia')
ax1.grid(True, alpha=0.3)

ax2.plot(k_values, silhouettes, 's-', color='coral', linewidth=2, markersize=8)
ax2.set_title('Silhouette Score', fontsize=13)
ax2.set_xlabel('Nombre de Clusters (k)')
ax2.set_ylabel('Score')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_k = k_values[np.argmax(silhouettes)]
print(f'\nK recommandé : {best_k} (meilleur silhouette = {max(silhouettes):.3f})')

## 7 — Exploration du Contenu des Clusters

In [ ]:
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]
top_words = {}

for cid in range(kmeans.n_clusters):
    top_terms = [feature_names[idx] for idx in order_centroids[cid, :10]]
    top_words[cid] = top_terms
    cluster_data = df_logs[df_logs['cluster_id'] == cid]
    size = len(cluster_data)

    print(f'\nCluster {cid} (Taille: {size})')
    print(f'  Top Termes: {', '.join(top_terms[:8])}')
    print('  Exemples :')
    for i, row in cluster_data.head(3).iterrows():
        print(f'    [{row["level"]}] {row["message"][:80]}')

    # Analyse par niveau
    level_dist = cluster_data['level'].value_counts()
    print(f'  Niveaux: {dict(level_dist)}')

## 8 — Analyse Approfondie & Classification des Clusters

In [ ]:
print('Analyse Approfondie des Clusters')
print('=' * 60)

for cid in range(kmeans.n_clusters):
    cluster_data = df_logs[df_logs['cluster_id'] == cid]
    size = len(cluster_data)
    terms = top_words[cid]

    # Déterminer le type
    error_terms = [t for t in terms[:5] if '500' in t or '502' in t or '503' in t or 'error' in t]
    warn_terms = [t for t in terms[:5] if '400' in t or '401' in t or '403' in t or '404' in t or 'warn' in t]

    if error_terms:
        ctype = 'ERREURS CRITIQUES'
        icon = '🔴'
    elif warn_terms:
        ctype = 'AVERTISSEMENTS'
        icon = '🟡'
    else:
        ctype = 'OPÉRATIONS NORMALES'
        icon = '🟢'

    # Variété des messages
    unique_msg = cluster_data['message'].nunique()
    variety = unique_msg / size if size > 0 else 0
    variety_desc = 'Haute' if variety > 0.7 else ('Moyenne' if variety > 0.3 else 'Faible')

    print(f'\n{icon} {ctype} — Cluster {cid} ({size} entrées)')
    print(f'  Top termes : {', '.join(terms[:8])}')
    print(f'  Variété : {variety_desc} ({variety:.1%} unique)')
    print(f'  Endpoints : {dict(cluster_data["endpoint"].value_counts().head(3))}')
    print(f'  Response time moyen : {cluster_data["response_time_ms"].mean():.1f} ms')

## 9 — Labeling & Interprétation des Clusters

In [ ]:
# Labels automatiques basés sur l'analyse
cluster_labels = {}
for cid in range(kmeans.n_clusters):
    terms = top_words[cid]
    cluster_data = df_logs[df_logs['cluster_id'] == cid]
    top_level = cluster_data['level'].mode().iloc[0] if len(cluster_data) > 0 else 'UNKNOWN'
    top_endpoint = cluster_data['endpoint'].mode().iloc[0] if len(cluster_data) > 0 else 'unknown'
    top_component = cluster_data['component'].mode().iloc[0] if len(cluster_data) > 0 else 'unknown'

    # Construire un label significatif
    error_flag = any('500' in t or '502' in t or '503' in t for t in terms[:5])
    warn_flag = any('400' in t or '401' in t or '403' in t for t in terms[:5])

    if error_flag:
        label = f'Erreurs Serveur ({top_component})'
    elif warn_flag:
        label = f'Accès Non Autorisés ({top_component})'
    else:
        label = f'Opérations {top_component.capitalize()} ({top_endpoint})'
    cluster_labels[cid] = label

df_logs['cluster_label'] = df_logs['cluster_id'].map(cluster_labels)

# Résumé
summary = df_logs.groupby(['cluster_id', 'cluster_label']).size().reset_index(name='count')
summary['percentage'] = (summary['count'] / len(df_logs) * 100).round(1)
summary = summary.sort_values('count', ascending=False)

print('Résumé des Clusters Labellisés :')
print('-' * 70)
print(f'{"ID":<4} {"Label":<40} {"Count":<8} {"%":<6}')
print('-' * 70)
for _, row in summary.iterrows():
    print(f'{row["cluster_id"]:<4} {row["cluster_label"]:<40} {row["count"]:<8} {row["percentage"]:<6}%')
print('-' * 70)
print(f'{"Total":<44} {len(df_logs):<8} {"100.0":<6}%')

## 10 — Visualisation des Clusters

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
sizes = [summary[summary['cluster_id'] == cid]['count'].values[0] for cid in range(kmeans.n_clusters)]
labels = [cluster_labels[cid] for cid in range(kmeans.n_clusters)]
colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12', '#9b59b6']
axes[0].pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Répartition des Clusters', fontsize=13)

# Bar chart par level
level_cluster = df_logs.groupby(['cluster_id', 'level']).size().unstack(fill_value=0)
level_cluster.plot(kind='bar', stacked=True, ax=axes[1], color=['#e74c3c', '#2ecc71', '#f39c12'])
axes[1].set_title('Niveaux de Log par Cluster', fontsize=13)
axes[1].set_xlabel('Cluster ID')
axes[1].set_ylabel('Nombre de logs')
axes[1].legend(title='Level')

plt.tight_layout()
plt.show()

## 11 — Documentation & Export

In [ ]:
# Construire le résumé final
log_cluster_summary = pd.DataFrame([
    {
        'ClusterID': cid,
        'Label': cluster_labels.get(cid, 'Unknown'),
        'TopTerms': ', '.join(top_words[cid][:8]),
        'ExampleLog': df_logs[df_logs['cluster_id'] == cid]['message'].iloc[0],
        'Count': int((df_logs['cluster_id'] == cid).sum()),
        'Percentage': round((df_logs['cluster_id'] == cid).sum() / len(df_logs) * 100, 1),
        'AvgResponseTime': round(df_logs[df_logs['cluster_id'] == cid]['response_time_ms'].mean(), 1)
    }
    for cid in range(kmeans.n_clusters)
])

log_cluster_summary = log_cluster_summary.sort_values('Count', ascending=False)

print('Résumé Complet des Clusters :')
display(log_cluster_summary)

# Export
output_path = os.path.join(PROJECT_DIR, 'datasets', 'log_clusters_summary.csv')
log_cluster_summary.to_csv(output_path, index=False)
print(f'\nExporté vers : {output_path}')

## 12 — Récapitulatif

In [ ]:
print('=' * 70)
print('  ANALYSE DES LOGS LOKI — TERMINÉE')
print('=' * 70)
print()
print(f'  Total logs traités : {len(df_logs):,}')
print(f'  Clusters identifiés : {kmeans.n_clusters}')
print(f'  Vocabulaire TF-IDF : {len(feature_names)} termes')
print(f'  K recommandé : {best_k}')
print()
print('Pipeline appliqué :')
print('  ✅ Chargement et exploration du dataset Loki')
print('  ✅ Nettoyage et extraction de patterns')
print('  ✅ Vectorisation TF-IDF')
print('  ✅ Exploration du vocabulaire')
print('  ✅ Clustering K-Means')
print('  ✅ Évaluation Elbow + Silhouette')
print('  ✅ Analyse approfondie des clusters')
print('  ✅ Labeling & interprétation')
print('  ✅ Visualisation')
print('  ✅ Export du résumé')
print()
print('=' * 70)